## Creating MCP- Langchain agent for accessing MongoDB 

In [51]:
import os 
from dotenv import load_dotenv 
load_dotenv() 
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


import warnings

warnings.filterwarnings("ignore",category=DeprecationWarning)

from langchain.chat_models import init_chat_model

gemma = init_chat_model(model="gemma4:latest", model_provider="ollama")

#llm = init_chat_model(model="qwen/qwen3-32b", model_provider="Groq")
llm_primary = init_chat_model(model="llama-3.3-70b-versatile", model_provider="Groq")
llm_fallback_1 = init_chat_model(model="gpt-5.4-nano", model_provider="OpenAI")
llm_fallback_2 = init_chat_model(model="gpt-5.4-mini", model_provider="OpenAI")


In [52]:
MONGODB_URI = os.getenv("MONGODB_URI")

In [53]:
from langchain_mcp_adapters.client import MultiServerMCPClient

## Connect your client with the MongoDB-MCP-server 

In [54]:
client = MultiServerMCPClient({
"mongodb":{
    "transport":"stdio",
    "command":"npx",
    "args":[
        "-y",
        "mongodb-mcp-server@latest",
        "--loggers",
        "stderr"
    ],
    "env":{
        "MDB_MCP_CONNECTION_STRING":MONGODB_URI
    }
},
})

In [55]:
tools_mongoDB = await client.get_tools()

Process group termination failed for PID 9634: [Errno 1] Operation not permitted, falling back to simple terminate


In [56]:
for tool in tools_mongoDB: 
    print(tool.name)

aggregate-db
aggregate
collection-indexes
collection-schema
collection-storage-size
connect
count
create-collection
create-index
db-stats
delete-many
disconnect
drop-collection
drop-database
drop-index
explain
export
find
insert-many
list-collections
list-connections
list-databases
mongodb-logs
rename-collection
update-many
list-knowledge-sources
search-knowledge


## Create Langchain agent with mcp_tools

In [57]:
from langchain.agents import create_agent

prompt="""You are a helpful mongodb assistant. 
use tools_mongoDB for connecting and acceesing mongodb database and answer based on user query.
"""

agent= create_agent(
    model=  llm_fallback_1, #gemma,
    tools=tools_mongoDB,
    system_prompt=prompt
)

## Test the agent

In [58]:
user_query= """How many documents are there in the collection?
 database: University, collection: students
 """

In [59]:
user_query= """Show me the document information with field information:
name:James Cercone
for database: University, collection: students
"""

In [60]:
user_query= """Modify the following record:
current name:James Cercone modify to name: James Chase 
for database: University, collection: students
"""

In [64]:
user_query= """Add the following new document:
name:Ethan Nawaz 
dept_name:Computer Science
gpa:3.99
credit_hours:112 
for database: University, collection: students
"""

In [67]:
user_query= """Delete the following document:
name:Ethan Nawaz  
for database: University, collection: students
"""

In [68]:
from langchain.messages import SystemMessage,HumanMessage

try:
    result = await agent.ainvoke({
        "messages":[
            SystemMessage(content="You a helpful assistant."),
            HumanMessage(content=user_query)
        ]
    })
except Exception as e:
    print(f"Error happened during invoke. {e}")

In [69]:
print(result["messages"][-1].content)

Deleted `1` document from **University.students** where **name = "Ethan Nawaz"**.
